# AI-Based Fake Job Posting Detection System
## Part 1: Exploratory Data Analysis (EDA)

This notebook covers the Exploratory Data Analysis phase of the Fake Job Posting Detection System. We analyze the properties of the Kaggle EMSCAD dataset (17,880 postings) to understand patterns, missing values, class distributions, and feature correlations with fraudulence.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

### 1. Load Dataset
We load the downloaded CSV file and check its shape and fields.

In [ ]:
data_path = "../data/fake_job_postings.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError("Dataset not found. Please run the download script first.")

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Class Imbalance (Target Distribution)
Let's see how many postings are labeled as genuine (`0`) and how many as fraudulent (`1`).

In [ ]:
counts = df["fraudulent"].value_counts()
percentages = df["fraudulent"].value_counts(normalize=True) * 100

print("Class Counts:")
print(counts)
print("\nClass Percentages:")
print(percentages)

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(x=counts.index, y=counts.values, palette=["#2ea44f", "#d73a49"], ax=ax)
ax.set_xticklabels(["Genuine (0)", "Fraudulent (1)"])
ax.set_title("Distribution of Job Postings")
ax.set_ylabel("Count")
plt.show()

### 3. Missing Value Analysis
We check which features have missing values. Many fields in job descriptions are optional.

In [ ]:
missing = df.isnull().mean() * 100
missing_df = pd.DataFrame({"Missing %": missing}).sort_values("Missing %", ascending=False)
print("Columns with missing values (percentage):")
print(missing_df[missing_df["Missing %"] > 0])

### 4. Correlation of Listing Completeness with Fraud
We investigate whether fraudulent listings are less complete (e.g., lack company logo, profile details, or screening questions) than genuine listings.

In [ ]:
# 1. Company Logo vs Fraud
logo_fraud = df.groupby("has_company_logo")["fraudulent"].mean() * 100
print("Fraud rate by company logo presence:")
print(logo_fraud)

# 2. Screening Questions vs Fraud
questions_fraud = df.groupby("has_questions")["fraudulent"].mean() * 100
print("\nFraud rate by screening questions presence:")
print(questions_fraud)

# 3. Telecommuting vs Fraud
remote_fraud = df.groupby("telecommuting")["fraudulent"].mean() * 100
print("\nFraud rate by remote work status:")
print(remote_fraud)

Let's visualize these percentages to see the contrast.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x=logo_fraud.index, y=logo_fraud.values, palette="Set2", ax=axes[0])
axes[0].set_xticklabels(["Missing Logo", "Has Logo"])
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud Rate by Logo Presence")

sns.barplot(x=questions_fraud.index, y=questions_fraud.values, palette="Set2", ax=axes[1])
axes[1].set_xticklabels(["No Questions", "Has Questions"])
axes[1].set_ylabel("Fraud Rate (%)")
axes[1].set_title("Fraud Rate by Questions Presence")

sns.barplot(x=remote_fraud.index, y=remote_fraud.values, palette="Set2", ax=axes[2])
axes[2].set_xticklabels(["On-site", "Remote"])
axes[2].set_ylabel("Fraud Rate (%)")
axes[2].set_title("Fraud Rate by Remote Status")

plt.tight_layout()
plt.show()

### 5. Vulnerable Sectors (Industries and Functions)
Let's see which business sectors and corporate departments are targeted most by job fraudsters.

In [ ]:
fake_df = df[df["fraudulent"] == 1]

print("Top 5 Industries for Fraudulent Postings:")
print(fake_df["industry"].value_counts().head(5))

print("\nTop 5 Job Functions for Fraudulent Postings:")
print(fake_df["function"].value_counts().head(5))

plt.figure(figsize=(10, 5))
sns.countplot(data=fake_df, y="industry", order=fake_df["industry"].value_counts().head(10).index, palette="Purples_r")
plt.title("Top 10 Industries Targeted by Scam Job Advertisements")
plt.xlabel("Count of Fraudulent Jobs")
plt.ylabel("Industry")
plt.show()

### 6. Text Character Length Distributions
Let's check if the length of the job description or company profile differs between fake and real postings.

In [ ]:
df["desc_len"] = df["description"].fillna("").apply(len)
df["profile_len"] = df["company_profile"].fillna("").apply(len)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df, x="fraudulent", y="desc_len", palette="Set2", showfliers=False, ax=axes[0])
axes[0].set_xticklabels(["Genuine (0)", "Fraudulent (1)"])
axes[0].set_title("Job Description Length (Characters)")
axes[0].set_ylabel("Length")

sns.boxplot(data=df, x="fraudulent", y="profile_len", palette="Set2", showfliers=False, ax=axes[1])
axes[1].set_xticklabels(["Genuine (0)", "Fraudulent (1)"])
axes[1].set_title("Company Profile Length (Characters)")
axes[1].set_ylabel("Length")

plt.show()

### Conclusion of EDA
1. **Imbalance**: The dataset is highly imbalanced with only ~4.8% fraudulent listings. This requires classification models that handle imbalance (via class weights or over/undersampling).
2. **Completeness**: Jobs without company logos or pre-screening questions have a significantly higher rate of being fraudulent. These categorical metadata fields are strong predictors.
3. **Sectors**: Administrative, Customer Service, Oil & Energy, and Financial Services are targeted heavily by job scams.
4. **Textual context**: Genuine jobs tend to have longer company profiles than fraudulent ones. Preprocessing these text columns using TF-IDF will serve as the core of our machine learning classifier.